# 🌦️ Vatavaran / GramMausam AI - Weather Forecasting Model Training

Welcome to the **Vatavaran Climate Studio** Machine Learning Notebook!

This notebook covers the full end-to-end ML pipeline for training weather prediction models:
1. 📥 **Data Fetching**: Download real historical weather observations (Temperature, Relative Humidity, Rain/Precipitation, Wind Speed, Surface Pressure) from the Open-Meteo API for target regions.
2. 🛠️ **Feature Engineering**: Time-series decomposition, temporal cyclical features (Sin/Cos day-of-year encoding), multi-day lagged observations, and rolling statistics.
3. 🤖 **Model Training**: Compare **Random Forest Regressor** and **XGBoost Regressor** models.
4. 📊 **Evaluation & Visualization**: Compute RMSE, MAE, $R^2$ metrics and plot predicted vs. actual weather trends.
5. 💾 **Model Export**: Save trained model artifacts (`.joblib`) for web application integration.

In [ ]:
# Step 1: Import Libraries
import os
import json
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb
import joblib

# Style config
plt.style.use('ggplot')
plt.rcParams['figure.figsize'] = (12, 6)
print("✅ All dependencies successfully imported!")

--- 
## 📥 Step 2: Fetch Historical Weather Data (Open-Meteo API)

In [ ]:
# Coordinates for Target Location (Default: Maharashtra - Lat: 18.5204, Lon: 73.8567)
LATITUDE = 18.5204
LONGITUDE = 73.8567

end_date = datetime.today().date()
start_date = end_date - timedelta(days=730) # Past 2 years of daily observations

url = f"https://archive-api.open-meteo.com/v1/archive?latitude={LATITUDE}&longitude={LONGITUDE}&start_date={start_date}&end_date={end_date}&daily=temperature_2m_max,temperature_2m_min,temperature_2m_mean,precipitation_sum,rain_sum,wind_speed_10m_max,relative_humidity_2m_mean&timezone=auto"

print(f"Fetching weather data from {start_date} to {end_date}...")
try:
    req = urllib.request.urlopen(url)
    data = json.loads(req.read().decode('utf-8'))
    daily_data = data['daily']
    df = pd.DataFrame(daily_data)
    df['time'] = pd.to_datetime(df['time'])
    df.set_index('time', inplace=True)
    print(f"Successfully fetched {len(df)} daily weather records!")
except Exception as e:
    print(f"API request failed ({e}). Generating realistic synthetic weather series for offline training...")
    dates = pd.date_range(start=start_date, end=end_date, freq='D')
    np.random.seed(42)
    n = len(dates)
    t_mean = 28 + 6 * np.sin(2 * np.pi * np.arange(n) / 365) + np.random.normal(0, 1.5, n)
    t_max = t_mean + np.random.uniform(4, 8, n)
    t_min = t_mean - np.random.uniform(4, 8, n)
    precip = np.maximum(0, np.random.choice([0, 0, 0, 5, 20, 50], size=n) * (1 + np.sin(2 * np.pi * (np.arange(n) - 150) / 365)))
    wind = np.random.uniform(5, 25, n)
    humidity = np.clip(50 + 25 * np.sin(2 * np.pi * (np.arange(n) - 150) / 365) + np.random.normal(0, 5, n), 20, 95)
    
    df = pd.DataFrame({
        'temperature_2m_max': t_max,
        'temperature_2m_min': t_min,
        'temperature_2m_mean': t_mean,
        'precipitation_sum': precip,
        'wind_speed_10m_max': wind,
        'relative_humidity_2m_mean': humidity
    }, index=dates)

df.head()

--- 
## 🛠️ Step 3: Feature Engineering & Time-Series Lags

In [ ]:
# Copy data for features
df_feat = df.copy()

# 1. Calendar Features & Cyclical Encoding
df_feat['day_of_year'] = df_feat.index.dayofyear
df_feat['month'] = df_feat.index.month
df_feat['sin_day'] = np.sin(2 * np.pi * df_feat['day_of_year'] / 365.25)
df_feat['cos_day'] = np.cos(2 * np.pi * df_feat['day_of_year'] / 365.25)

# 2. Lag Features (Previous 1, 2, 3 and 7 days weather)
for col in ['temperature_2m_mean', 'temperature_2m_max', 'precipitation_sum', 'relative_humidity_2m_mean']:
    for lag in [1, 2, 3, 7]:
        df_feat[f'{col}_lag_{lag}'] = df_feat[col].shift(lag)

# 3. Rolling Statistics (3-day and 7-day moving averages)
for col in ['temperature_2m_mean', 'relative_humidity_2m_mean', 'precipitation_sum']:
    df_feat[f'{col}_roll_3_mean'] = df_feat[col].shift(1).rolling(window=3).mean()
    df_feat[f'{col}_roll_7_mean'] = df_feat[col].shift(1).rolling(window=7).mean()
    df_feat[f'{col}_roll_7_std'] = df_feat[col].shift(1).rolling(window=7).std()

# Target Variable: Predict Next Day's Mean Temperature (t+1)
df_feat['target_temp_mean_next_day'] = df_feat['temperature_2m_mean'].shift(-1)

# Drop NaNs created by lagging/rolling/shifting
df_cleaned = df_feat.dropna()

print(f"Features shape: {df_cleaned.shape}")
df_cleaned[['temperature_2m_mean', 'target_temp_mean_next_day', 'sin_day', 'cos_day', 'temperature_2m_mean_lag_1']].head()

--- 
## 🤖 Step 4: Model Training & Hyperparameter Setup

In [ ]:
# Separate Features (X) and Target (y)
X = df_cleaned.drop(columns=['target_temp_mean_next_day'])
y = df_cleaned['target_temp_mean_next_day']

# Time-series split (Sequential split to avoid data leakage)
split_idx = int(len(X) * 0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

print(f"Training samples: {len(X_train)} | Testing samples: {len(X_test)}")

# Initialize Models
rf_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
xgb_model = xgb.XGBRegressor(n_estimators=100, learning_rate=0.05, max_depth=5, random_state=42)

# Fit Models
print("Training Random Forest Regressor...")
rf_model.fit(X_train, y_train)

print("Training XGBoost Regressor...")
xgb_model.fit(X_train, y_train)

print("✅ Training Complete!")

--- 
## 📊 Step 5: Model Evaluation & Predictions Plot

In [ ]:
rf_preds = rf_model.predict(X_test)
xgb_preds = xgb_model.predict(X_test)

def print_metrics(model_name, y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    print(f"📌 {model_name} Metrics:")
    print(f"   - RMSE (Root Mean Sq Error) : {rmse:.3f} °C")
    print(f"   - MAE  (Mean Abs Error)     : {mae:.3f} °C")
    print(f"   - R² Score                  : {r2:.3f}")
    return rmse, mae, r2

print_metrics("Random Forest", y_test, rf_preds)
print("-" * 40)
print_metrics("XGBoost", y_test, xgb_preds)

# Plot Predictions vs Actual
plt.figure(figsize=(14, 6))
plt.plot(y_test.index, y_test.values, label='Actual Temperature (°C)', color='black', alpha=0.8, linewidth=1.5)
plt.plot(y_test.index, rf_preds, label='Random Forest Forecast', color='dodgerblue', linestyle='--', alpha=0.9)
plt.plot(y_test.index, xgb_preds, label='XGBoost Forecast', color='crimson', linestyle=':', alpha=0.9)
plt.title('Vatavaran Weather Forecasting: Actual vs Predicted Next-Day Mean Temperature')
plt.xlabel('Date')
plt.ylabel('Temperature (°C)')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

--- 
## 💾 Step 6: Feature Importance & Saving Model Artifacts

In [ ]:
# Top 10 Feature Importances (XGBoost)
importances = pd.Series(xgb_model.feature_importances_, index=X.columns).sort_values(ascending=False)

plt.figure(figsize=(10, 5))
importances.head(10).plot(kind='barh', color='teal').invert_yaxis()
plt.title('Top 10 Feature Importances (XGBoost Weather Predictor)')
plt.xlabel('Relative Importance')
plt.tight_layout()
plt.show()

# Save Models & Feature Schema
os.makedirs('models', exist_ok=True)
joblib.dump(xgb_model, 'models/weather_xgb_model.joblib')
joblib.dump(rf_model, 'models/weather_rf_model.joblib')
with open('models/feature_columns.json', 'w') as f:
    json.dump(list(X.columns), f)

print("💾 Models saved successfully in ./models/ directory!")